In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, sys, math, random, datetime, json, gc, time
import numpy as np
import torch
from torch.utils.data import DataLoader, ConcatDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ================= 기본 설정 =================
# 프로젝트 경로(필요 시 수정)
current_dir = os.path.dirname(os.path.abspath(''))
two_up_dir = os.path.dirname(os.path.dirname(current_dir))
if two_up_dir not in sys.path:
    sys.path.append(two_up_dir)

# 모델/데이터셋/로스 임포트
from TinyCenterSpeed.src.models.CenterSpeed import (
    CenterSpeedDense, CenterSpeedDenseResidual, CenterSpeedDenseCBAM, CenterSpeedDenseBottleneck
)
from TinyCenterSpeed.dataset.CenterSpeed_dataset import CenterSpeedDataset, RandomRotation, RandomFlip
from TinyCenterSpeed.src.models.losses import *  # 내부 함수 사용 시

# -------- W&B 설정 --------
use_wandb = True
project_name = "TinyCenterSpeed_Dense_real_data"
run_name = "train_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    import wandb
    if use_wandb:
        if os.environ.get("WANDB_MODE","").lower() == "offline":
            wandb.init(project=project_name, name=run_name, mode="offline")
        else:
            try:
                wandb.init(project=project_name, name=run_name)
            except Exception:
                wandb.init(project=project_name, name=run_name, mode="offline")
except Exception as e:
    print(f"[wandb] 사용 불가: {type(e).__name__}: {e}")
    use_wandb = False
    wandb = None

# -------- 재현성/디바이스 --------
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.backends.cudnn.benchmark = True


wandb: Currently logged in as: whdaudpark (whdaudpark-dongguk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


In [2]:

# ================= 경로 설정 =================
# 기존 데이터 구조 유지 (필요 시 변경)
# train_obj_path  = "/home/harry/Downloads/lidarscanbag/free"
# train_free_path = "/home/harry/Downloads/lidarscanbag/free_csv"
# val_obj_path    = "/home/harry/Downloads/lidarscanbag/free"
# val_free_path   = "/home/harry/Downloads/lidarscanbag/free_csv"


# ================= real data =================
train_obj_path  = "/home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs"
train_free_path = "/home/harry/sim_ws/src/f1tenth_gym_ros/real_train_free"
val_obj_path    = "/home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs"
val_free_path   = "/home/harry/sim_ws/src/f1tenth_gym_ros/real_train_free"

# ================= 하이퍼파라미터 =================
image_size    = 128     # CenterSpeedDataset와 일치
pixelsize     = 0.1     # m/pixel
sigma_px      = 1.0     # 가우시안 σ (px)
epochs        = 60      # 이어 학습 에폭 수(새로 추가된 데이터 규모에 맞춰 조절)
batch_size    = 32
learning_rate = 5e-4    # 처음 학습 시
resume_lr     = 1e-4    # 이어 학습 시(권장: 더 낮게 시작)

# 이어학습(파인튜닝) 관련
RESUME_CKPT = "/home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/0_best_objfree_trainfree52497_20250818_170557_epoch_14_loss_088283.pt"
EPOCH_OFFSET = 14  # 기존 가중치의 마지막 에폭(파일명에 14가 있으므로 이어붙임 목적)

# DataLoader 최적화
NUM_WORKERS = min(os.cpu_count() or 4, 8)
PIN_MEMORY  = torch.cuda.is_available()
PERSIST     = NUM_WORKERS > 0

print({
    "image_size": image_size,
    "pixelsize": pixelsize,
    "sigma_px": sigma_px,
    "epochs(ft)": epochs,
    "batch_size": batch_size,
    "lr_init": learning_rate,
    "lr_resume": resume_lr,
    "workers": NUM_WORKERS
})

# --- W&B 메트릭 정의/환경 기록 ---
if 'wandb' in globals() and wandb and use_wandb:
    try:
        wandb.define_metric("epoch_total")     # 누적 에폭 번호
        wandb.define_metric("lr", step_metric="epoch_total")
    except Exception:
        pass
    try:
        wandb.config.update({
            "image_size": image_size,
            "pixelsize": pixelsize,
            "sigma_px": sigma_px,
            "epochs_ft": epochs,
            "batch_size": batch_size,
            "lr_init": learning_rate,
            "lr_resume": resume_lr,
            "optimizer": "Adam",
            "scheduler": "ReduceLROnPlateau(factor=0.5, patience=8, threshold=0.005)",
            "resume_ckpt": RESUME_CKPT,
            "epoch_offset": EPOCH_OFFSET,
        }, allow_val_change=True)
    except Exception:
        pass

# ================= 데이터셋/로더 =================
# 객체 없음 데이터에서 GT를 0으로 강제하는 래퍼
class ZeroTargetWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        inputs, gts, data_vec, dense_feats, is_free = self.base[idx]
        # heatmap, dense 모두 0으로
        gts = torch.zeros_like(gts)
        dense_feats = torch.zeros_like(dense_feats)
        return inputs, gts, data_vec, dense_feats, is_free

from torchvision import transforms as T
transform = T.Compose([
    RandomRotation(45, image_size=image_size),
    RandomFlip(0.5),
])

def make_dataset(root_dir, use_transform=False):
    ds = CenterSpeedDataset(dataset_path=root_dir, transform=(transform if use_transform else None), dense=True)
    try: ds.change_image_size(image_size)
    except Exception: ds.image_size = image_size
    try: ds.change_pixel_size(pixelsize)
    except Exception: ds.pixelsize = pixelsize
    ds.sx = sigma_px; ds.sy = sigma_px
    return ds

# Train
train_obj_dataset  = make_dataset(train_obj_path, use_transform=False)
train_free_dataset = make_dataset(train_free_path, use_transform=False)
train_free_dataset = ZeroTargetWrapper(train_free_dataset)

train_dataset = ConcatDataset([train_obj_dataset, train_free_dataset])

# Val
val_obj_dataset  = make_dataset(val_obj_path, use_transform=False)
val_free_dataset = make_dataset(val_free_path, use_transform=False)
val_free_dataset = ZeroTargetWrapper(val_free_dataset)

val_dataset = ConcatDataset([val_obj_dataset, val_free_dataset])

print("train_obj:", len(train_obj_dataset), 
      "train_free:", len(train_free_dataset), 
      "=> train_total:", len(train_dataset))
print("val_obj:", len(val_obj_dataset), 
      "val_free:", len(val_free_dataset), 
      "=> val_total:", len(val_dataset))

def worker_init_fn(_):
    try:
        import torch, os
        torch.set_num_threads(1)
        os.environ.setdefault("OMP_NUM_THREADS", "1")
        os.environ.setdefault("MKL_NUM_THREADS", "1")
    except Exception:
        pass

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST,
    prefetch_factor=4,
    drop_last=True,
    worker_init_fn=worker_init_fn,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST,
    prefetch_factor=2,
    drop_last=False,
    worker_init_fn=worker_init_fn,
)


{'image_size': 128, 'pixelsize': 0.1, 'sigma_px': 1.0, 'epochs(ft)': 60, 'batch_size': 32, 'lr_init': 0.0005, 'lr_resume': 0.0001, 'workers': 8}
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs/redbull_obs1_box.csv
Entries     :  1580
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs/redbull_obs1_car.csv
Entries     :  1984
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs/redbull_obs2_car.csv
Entries     :  2253
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs/redbull_obs2_laying_box.csv
Entries     :  1714
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/real_train_obs/redbull_obs2_standing_box.csv
Entries     :  2068
Total rows :  9599
File index :  [(0, 1580), (1580, 3564), (3564, 5817), (5817, 7531), (7531, 9599)]
Image size   -> 128
Origin offset-> 6.4
Pixel size   -> 0.1
Origin offset-> 6.4
Total rows :  0
File index :  []
Image size   -> 128
Origin offset-> 6.4
Pixel size   -> 0.1
Origin offset->

In [3]:

# ================= 모델/옵티마이저 =================
model = CenterSpeedDense(input_channels=4, image_size=image_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # 초기 LR로 생성
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=8, threshold=0.005)

def get_lr(optim):
    return optim.param_groups[0]["lr"]

# ====== 체크포인트 로드(이어 학습) ======
def _strip_module_prefix(state_dict):
    if any(k.startswith("module.") for k in state_dict.keys()):
        return {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    return state_dict

if RESUME_CKPT and os.path.isfile(RESUME_CKPT):
    print(f"[Resume] Loading weights from: {RESUME_CKPT}")
    state = torch.load(RESUME_CKPT, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]  # 혹시 전체 딕셔너리로 저장된 경우 대비
    state = _strip_module_prefix(state)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[Resume] loaded. missing_keys={len(missing)}, unexpected_keys={len(unexpected)}")
    if missing:   print("  - missing:", missing[:10], "..." if len(missing)>10 else "")
    if unexpected: print("  - unexpected:", unexpected[:10], "..." if len(unexpected)>10 else "")
    # 이어학습은 안정적으로 가도록 LR 낮춤
    for g in optimizer.param_groups:
        g["lr"] = resume_lr
else:
    print("[Resume] No checkpoint found or path empty. Training from scratch weights.")

print("Model/optimizer/loss 준비 완료")

# ================== 손실 함수 ==================
# 출력 [B,4,H,W], gts [B,H,W], dense [B,H,W,3]
def dense_loss(output, gt_heatmap, gt_dense_data, is_free, alpha=0.99, decay=1.0):
    preds = output.permute(0,2,3,1)  # [B,H,W,4]
    w = gt_heatmap.unsqueeze(-1)     # [B,H,W,1]
    loss_occ   = (alpha     * (1 + w) * (preds[...,0:1] - gt_heatmap.unsqueeze(-1))**2).sum()
    loss_dense = ((1-alpha) * (1 + w) * (preds[...,1:]  - gt_dense_data)**2).sum()
    batch_size = output.shape[0]
    return (loss_occ + loss_dense) / batch_size

# ================== 학습 루프 ==================
best_val = float('inf')
train_hist, val_hist = [], []

save_dir = "/home/harry/ros2_ws/src/TinyCenterSpeed/src/pt"
os.makedirs(save_dir, exist_ok=True)

for i_epoch in range(1, epochs+1):
    epoch_total = EPOCH_OFFSET + i_epoch  # 이어붙이는 누적 에폭 표기
    model.train()
    running = 0.0
    for batch in train_loader:
        inputs, gts, data_vec, dense_feats, is_free = batch
        inputs      = inputs.to(device, non_blocking=True)
        gts         = gts.to(device, non_blocking=True)
        dense_feats = dense_feats.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        out = model(inputs)
        loss = dense_loss(out, gts, dense_feats, is_free, alpha=0.99)
        loss.backward()
        optimizer.step()
        running += loss.item()

    train_loss = running / max(1, len(train_loader))
    train_hist.append(train_loss)

    # Validation
    model.eval()
    v_running = 0.0
    with torch.no_grad():
        for batch in val_loader:
            inputs, gts, data_vec, dense_feats, is_free = batch
            inputs      = inputs.to(device, non_blocking=True)
            gts         = gts.to(device, non_blocking=True)
            dense_feats = dense_feats.to(device, non_blocking=True)
            out = model(inputs)
            v_loss = dense_loss(out, gts, dense_feats, is_free, alpha=0.99)
            v_running += v_loss.item()

    val_loss = v_running / max(1, len(val_loader)) if len(val_loader)>0 else train_loss
    val_hist.append(val_loss)

    # 스케줄러 업데이트
    scheduler.step(val_loss)

    # W&B 로그
    if 'wandb' in globals() and wandb and use_wandb:
        try:
            wandb.log({
                "epoch_total": epoch_total,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "lr": get_lr(optimizer)
            }, step=epoch_total)
        except Exception:
            pass

    now_h = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    print(f"[{i_epoch:03d}/{epochs}] (total {epoch_total}) train={train_loss:.6f} | val={val_loss:.6f} | lr={get_lr(optimizer):.6g}")

    # 베스트 저장
    if val_loss < best_val or (epoch_total % 10) == 0:
        best_val = val_loss
        best_path = os.path.join(
            save_dir,
            f"CenterSpeedDense_transfer_no_free_t_{now_h}_epoch{EPOCH_OFFSET}_epochTotal_{epoch_total}_val_{best_val:.6f}.pt"
        )
        torch.save(model.state_dict(), best_path)
        print(f"  ↳ Best model saved: {best_path}")
    
# 마지막 모델 저장
last_path = os.path.join(
    save_dir,
    f"CenterSpeedDense_transfer_no_free_t_{now_h}_epoch{EPOCH_OFFSET}_epochTotal_{epoch_total}.pt"
)
torch.save(model.state_dict(), last_path)
print(f"Training finished. Model saved at {last_path}")


[Resume] Loading weights from: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/0_best_objfree_trainfree52497_20250818_170557_epoch_14_loss_088283.pt
[Resume] loaded. missing_keys=0, unexpected_keys=0
Model/optimizer/loss 준비 완료
[001/60] (total 15) train=2.092371 | val=1.538965 | lr=0.0001
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedDense_transfer_no_free_t_20250820_182209_epoch14_epochTotal_15_val_1.538965.pt
[002/60] (total 16) train=1.340007 | val=1.172332 | lr=0.0001
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedDense_transfer_no_free_t_20250820_182216_epoch14_epochTotal_16_val_1.172332.pt
[003/60] (total 17) train=1.102535 | val=1.012468 | lr=0.0001
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedDense_transfer_no_free_t_20250820_182224_epoch14_epochTotal_17_val_1.012468.pt
[004/60] (total 18) train=0.978138 | val=0.916693 | lr=0.0001
  ↳ Best model saved: /home/harry/ros2_ws/src/Tin

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x74ece3a16500>> (for post_run_cell), with arguments args (<ExecutionResult object at 74ecd2c7d9f0, execution_count=3 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 74ecd2c7da20, raw_cell="
# ================= 모델/옵티마이저 =================
mo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/harry/ros2_ws/src/TinyCenterSpeed/src/train/1_train_transfer.ipynb#W2sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe